# File 05 LR — Lighting-Robust Segmented Training
Train EfficientNet-B0 with CLAHE + RandomShadow. No test evaluation.


In [1]:
%run 04_diabetes_segmented_preprocessing_lighting_robust.ipynb

C:\Users\CompuMark\AppData\Roaming\Python\Python313\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Device: cuda
Lighting correction defined.
Loaded 2750 rows. Label mapping PASS.
Train:1930 Val:408 Test:412
train: [32, 3, 224, 224], labels=[0, 1]
val: [32, 3, 224, 224], labels=[1]
test: [32, 3, 224, 224], labels=[1]
Lighting comparison saved.
FILE 04 LR HANDOFF
Status: PASS
Manifest: D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv
Train:1930 Val:408 Test:412
Lighting: CLAHE LAB L-channel
Batch: 32 Workers: 0 Device: cuda
Batch sanity: PASS



In [2]:
import json, time
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torchvision.models import efficientnet_b0

TRAIN_OUT=Path(r'D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust')
TRAIN_OUT.mkdir(exist_ok=True)


## RandomShadow Augmentation (train only)


In [3]:
import random as pyrandom

class RandomShadow:
    """Soft polygon shadow, train only."""
    def __init__(self, p=0.35, strength_range=(0.55,0.85)):
        self.p=p; self.strength_range=strength_range
    def __call__(self, img_pil):
        if pyrandom.random()>self.p: return img_pil
        arr=np.array(img_pil).astype(np.float32)
        h,w=arr.shape[:2]
        shadow_mask=np.zeros((h,w),dtype=np.float32)
        x0,y0=pyrandom.randint(0,w),0
        x1,y1=pyrandom.randint(0,w),h
        for y in range(h):
            xb=int(x0+(x1-x0)*(y/h))
            if pyrandom.random()<0.5:
                shadow_mask[y,:xb]=1
            else:
                shadow_mask[y,xb:]=1
        sigma=max(h,w)//8
        shadow_mask=cv2.GaussianBlur(shadow_mask,(0,0),sigma)
        shadow_mask=shadow_mask/shadow_mask.max() if shadow_mask.max()>0 else shadow_mask
        strength=pyrandom.uniform(*self.strength_range)
        shadow_mask=1.0-shadow_mask*strength*(0.55)
        arr=arr*shadow_mask[:,:,np.newaxis]
        arr=np.clip(arr,0,255).astype(np.uint8)
        return Image.fromarray(arr)

print('RandomShadow defined.')


RandomShadow defined.


## Rebuild Train Transform with RandomShadow


In [4]:
train_transform_lr=T.Compose([
    T.Lambda(correct_lighting_clahe),
    T.Lambda(pad_square),
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(7),
    T.RandomAffine(0,translate=(0.03,0.03),scale=(0.95,1.05)),
    T.Lambda(RandomShadow(p=0.35)),
    T.ColorJitter(brightness=0.12,contrast=0.12,saturation=0.06,hue=0.01),
    T.ToTensor(),
    T.Normalize(mean=MEAN,std=STD),
])

ds_train_lr=SegDataset(df_train,train_transform_lr)
dl_train_lr=DataLoader(ds_train_lr,BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)
print('LR train dataloader ready.')


LR train dataloader ready.


## Config and Model


In [5]:
CONFIG={'model':'EfficientNet-B0','pretrained':'DEFAULT','image_size':IMG_SIZE,'batch_size':BATCH_SIZE,'optimizer':'AdamW','loss':'BCEWithLogitsLoss','device':str(DEVICE),'mixed_precision':torch.cuda.is_available(),'phase1_epochs':10,'phase1_lr':1e-3,'phase2_epochs':20,'phase2_lr':1e-4,'early_stop_patience':8,'scheduler':'ReduceLROnPlateau','seed':SEED,'lighting_correction':'CLAHE_LAB_L_clip1.5','random_shadow':'p=0.35 strength=0.55-0.85'}
with open(TRAIN_OUT/'05_lr_training_config.json','w') as f: json.dump(CONFIG,f,indent=2)

model=efficientnet_b0(weights='DEFAULT')
model.classifier=nn.Sequential(nn.Dropout(0.3),nn.Linear(model.classifier[1].in_features,1))
model=model.to(DEVICE)
total=sum(p.numel() for p in model.parameters()); trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total:{total:,} Trainable:{trainable:,}')
with open(TRAIN_OUT/'05_lr_model_summary.txt','w') as f: f.write(f'EfficientNet-B0\nTotal:{total:,}\nTrainable:{trainable:,}\n')


Total:4,008,829 Trainable:4,008,829


## Training Functions


In [6]:
criterion=nn.BCEWithLogitsLoss()
scaler=GradScaler() if CONFIG['mixed_precision'] else None

def compute_metrics(y_true,y_prob,thr=0.5):
    y_pred=(y_prob>=thr).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
    recall=tp/(tp+fn) if (tp+fn)>0 else 0; spec=tn/(tn+fp) if (tn+fp)>0 else 0
    ppv=tp/(tp+fp) if (tp+fp)>0 else 0; npv=tn/(tn+fn) if (tn+fn)>0 else 0
    f1=2*ppv*recall/(ppv+recall) if (ppv+recall)>0 else 0
    bacc=(recall+spec)/2; acc=(tp+tn)/(tp+tn+fp+fn)
    return {'diabetes_recall':recall,'specificity':spec,'ppv':ppv,'npv':npv,'f1':f1,'balanced_accuracy':bacc,'accuracy':acc,'fnr':fn/(tp+fn) if (tp+fn)>0 else 0,'roc_auc':roc_auc_score(y_true,y_prob),'pr_auc':average_precision_score(y_true,y_prob),'brier':brier_score_loss(y_true,y_prob),'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn)}

def find_best_threshold(y_true,y_prob):
    best_s,best_t=-1,0.5
    for t in np.arange(0.05,0.96,0.05):
        m=compute_metrics(y_true,y_prob,t)
        s=0.45*m['diabetes_recall']+0.25*m['specificity']+0.20*m['pr_auc']+0.10*m['balanced_accuracy']
        if s>best_s: best_s,best_t=s,t
    return best_t,best_s

def train_epoch(model,loader,opt):
    model.train(); loss_sum=0
    for imgs,labels in loader:
        imgs,labels=imgs.to(DEVICE),labels.float().to(DEVICE).unsqueeze(1)
        opt.zero_grad()
        if scaler:
            with autocast(): out=model(imgs); loss=criterion(out,labels)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            out=model(imgs); loss=criterion(out,labels); loss.backward(); opt.step()
        loss_sum+=loss.item()*imgs.size(0)
    return loss_sum/len(loader.dataset)

def eval_epoch(model,loader):
    model.eval(); loss_sum=0; yt,yp=[],[]
    with torch.no_grad():
        for imgs,labels in loader:
            imgs,lt=imgs.to(DEVICE),labels.float().to(DEVICE).unsqueeze(1)
            out=model(imgs); loss=criterion(out,lt); loss_sum+=loss.item()*imgs.size(0)
            yt.extend(labels.cpu().numpy()); yp.extend(torch.sigmoid(out).cpu().numpy().flatten())
    return loss_sum/len(loader.dataset),np.array(yt),np.array(yp)


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=GradScaler() if CONFIG['mixed_precision'] else None


## Phase 1


In [7]:
for p in model.features.parameters(): p.requires_grad=False
opt=optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=CONFIG['phase1_lr'])
sch=optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=0.5,patience=3)
history=[]; best_score,best_epoch,patience_ctr=-1,0,0
print('=== PHASE 1 ===')
for epoch in range(1,CONFIG['phase1_epochs']+1):
    t0=time.time(); tl=train_epoch(model,dl_train_lr,opt); vl,yt,yp=eval_epoch(model,dl_val)
    m05=compute_metrics(yt,yp,0.5); bt,_=find_best_threshold(yt,yp); mt=compute_metrics(yt,yp,bt)
    score=0.45*mt['diabetes_recall']+0.25*mt['specificity']+0.20*mt['pr_auc']+0.10*mt['balanced_accuracy']
    history.append({'phase':1,'epoch':epoch,'train_loss':tl,'val_loss':vl,'screening_score':score,'best_threshold':bt,'lr':opt.param_groups[0]['lr'],'time_s':time.time()-t0,'threshold_0.5':m05,'threshold_tuned':mt})
    sch.step(vl)
    if score>best_score:
        best_score,best_epoch,patience_ctr=score,epoch,0
        torch.save(model.state_dict(),TRAIN_OUT/'best_segmented_lighting_robust_model.pth')
        print(f'Epoch {epoch}: val_loss={vl:.4f} score={score:.4f} [BEST]')
    else:
        patience_ctr+=1
        print(f'Epoch {epoch}: val_loss={vl:.4f} score={score:.4f}')
    if patience_ctr>=CONFIG['early_stop_patience']: print('Early stop'); break
print(f'Phase 1 done. Best:{best_epoch} score:{best_score:.4f}')


=== PHASE 1 ===


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): out=model(imgs); loss=criterion(out,labels)


Epoch 1: val_loss=0.4322 score=0.8808 [BEST]


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): out=model(imgs); loss=criterion(out,labels)


Epoch 2: val_loss=0.3669 score=0.8893 [BEST]


...

Epoch 8: val_loss=0.3087 score=0.9060


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): out=model(imgs); loss=criterion(out,labels)


Epoch 9: val_loss=0.2962 score=0.9124 [BEST]


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): out=model(imgs); loss=criterion(out,labels)


Epoch 10: val_loss=0.3153 score=0.9064
Phase 1 done. Best:9 score:0.9124


## Phase 2


In [8]:
for name,p in model.features.named_parameters():
    if any(f'_{i}' in name for i in [6,7,8]): p.requires_grad=True
opt=optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=CONFIG['phase2_lr'])
sch=optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=0.5,patience=3)
best_score2,patience_ctr2=-1,0
print('=== PHASE 2 ===')
for epoch in range(1,CONFIG['phase2_epochs']+1):
    t0=time.time(); tl=train_epoch(model,dl_train_lr,opt); vl,yt,yp=eval_epoch(model,dl_val)
    m05=compute_metrics(yt,yp,0.5); bt,_=find_best_threshold(yt,yp); mt=compute_metrics(yt,yp,bt)
    score=0.45*mt['diabetes_recall']+0.25*mt['specificity']+0.20*mt['pr_auc']+0.10*mt['balanced_accuracy']
    history.append({'phase':2,'epoch':epoch,'train_loss':tl,'val_loss':vl,'screening_score':score,'best_threshold':bt,'lr':opt.param_groups[0]['lr'],'time_s':time.time()-t0,'threshold_0.5':m05,'threshold_tuned':mt})
    sch.step(vl)
    if score>best_score2:
        best_score2,patience_ctr2=score,0
        torch.save(model.state_dict(),TRAIN_OUT/'best_segmented_lighting_robust_model.pth')
        best_epoch=epoch; best_score=score
        print(f'Epoch {epoch}: val_loss={vl:.4f} score={score:.4f} [BEST]')
    else:
        patience_ctr2+=1
        print(f'Epoch {epoch}: val_loss={vl:.4f} score={score:.4f}')
    if patience_ctr2>=CONFIG['early_stop_patience']: print('Early stop'); break
torch.save(model.state_dict(),TRAIN_OUT/'last_segmented_lighting_robust_model.pth')
print(f'Phase 2 done.')


=== PHASE 2 ===


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): out=model(imgs); loss=criterion(out,labels)


Epoch 1: val_loss=0.2952 score=0.9156 [BEST]


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): out=model(imgs); loss=criterion(out,labels)


Epoch 2: val_loss=0.3074 score=0.9111


...

Epoch 14: val_loss=0.2861 score=0.9156


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): out=model(imgs); loss=criterion(out,labels)


Epoch 15: val_loss=0.3036 score=0.9146


C:\Users\CompuMark\AppData\Local\Temp\ipykernel_27928\3622816162.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): out=model(imgs); loss=criterion(out,labels)


Epoch 16: val_loss=0.3006 score=0.9115
Early stop
Phase 2 done.


## Save Results


In [9]:
rows=[]
for h in history:
    row={'phase':h['phase'],'epoch':h['epoch'],'train_loss':h['train_loss'],'val_loss':h['val_loss'],'screening_score':h['screening_score'],'lr':h['lr'],'time_s':h['time_s'],'best_threshold':h['best_threshold']}
    for k,v in h['threshold_0.5'].items(): row[f't05_{k}']=v
    for k,v in h['threshold_tuned'].items(): row[f'tuned_{k}']=v
    rows.append(row)
pd.DataFrame(rows).to_csv(TRAIN_OUT/'05_lr_epoch_metrics.csv',index=False)

best_rec=history[best_epoch-1]
best_summary={'best_epoch':best_epoch,'screening_score':float(best_score),'best_threshold':float(best_rec['best_threshold']),'val_loss':float(best_rec['val_loss']),'diabetes_recall_tuned':float(best_rec['threshold_tuned']['diabetes_recall']),'specificity_tuned':float(best_rec['threshold_tuned']['specificity']),'roc_auc':float(best_rec['threshold_tuned']['roc_auc']),'pr_auc':float(best_rec['threshold_tuned']['pr_auc'])}
with open(TRAIN_OUT/'05_lr_best_checkpoint_summary.json','w') as f: json.dump(best_summary,f,indent=2)

model.load_state_dict(torch.load(TRAIN_OUT/'best_segmented_lighting_robust_model.pth'))
_,yt,yp=eval_epoch(model,dl_val)
pd.DataFrame({'y_true':yt,'y_prob':yp}).to_csv(TRAIN_OUT/'05_lr_validation_predictions_best.csv',index=False)

from sklearn.metrics import confusion_matrix as cmx
y_pred=(yp>=best_rec['best_threshold']).astype(int)
cm=cmx(yt,y_pred,labels=[0,1])
fig,ax=plt.subplots(figsize=(6,6)); ax.matshow(cm,cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=14)
ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['ND','D']); ax.set_yticklabels(['ND','D'])
plt.tight_layout(); plt.savefig(TRAIN_OUT/'05_lr_confusion_matrix_val_best.png',dpi=100); plt.close()

df_h=pd.DataFrame(rows)
fig,axes=plt.subplots(2,2,figsize=(12,10))
axes[0,0].plot(df_h['epoch'],df_h['train_loss'],label='train'); axes[0,0].plot(df_h['epoch'],df_h['val_loss'],label='val'); axes[0,0].set_title('Loss'); axes[0,0].legend()
axes[0,1].plot(df_h['epoch'],df_h['tuned_diabetes_recall']); axes[0,1].set_title('Recall')
axes[1,0].plot(df_h['epoch'],df_h['tuned_roc_auc']); axes[1,0].set_title('ROC AUC')
axes[1,1].plot(df_h['epoch'],df_h['screening_score']); axes[1,1].set_title('Screening Score')
plt.tight_layout(); plt.savefig(TRAIN_OUT/'05_lr_training_curves.png',dpi=100); plt.close()

# Compare to old File 05 if available
old_path=Path(r'D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training\05_segmented_best_checkpoint_summary.json')
comp_str=''
if old_path.exists():
    with open(old_path) as f: old=json.load(f)
    comp_str=f'\nComparison to previous File 05:\nOld recall:{old.get("diabetes_recall_tuned","N/A")} New recall:{best_summary["diabetes_recall_tuned"]:.4f}\nOld ROC:{old.get("roc_auc","N/A")} New ROC:{best_summary["roc_auc"]:.4f}'

handoff=f'FILE 05 LR HANDOFF\nStatus: COMPLETE\nSource: {MANIFEST_PATH}\nLighting: CLAHE LAB L-channel\nRandomShadow: train only (p=0.35)\nTrain:{len(df_train)} Val:{len(df_val)} Test:{len(df_test)} (held out)\nBest epoch:{best_epoch} score:{best_score:.4f} threshold:{best_rec["best_threshold"]:.3f}\nRecall:{best_summary["diabetes_recall_tuned"]:.4f} Spec:{best_summary["specificity_tuned"]:.4f} ROC:{best_summary["roc_auc"]:.4f}\nTest NOT evaluated.{comp_str}\n'
with open(TRAIN_OUT/'05_lr_handoff_summary.txt','w') as f: f.write(handoff)
print(handoff)


FILE 05 LR HANDOFF
Status: COMPLETE
Source: D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv
Lighting: CLAHE LAB L-channel
RandomShadow: train only (p=0.35)
Train:1930 Val:408 Test:412 (held out)
Best epoch:8 score:0.9186 threshold:0.500
Recall:0.9397 Spec:0.8182 ROC:0.9540
Test NOT evaluated.

